In [1]:
import os
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. FILE CONFIGURATION & PATH SETUP
# -----------------------------------------------------------------------------
FILE_PATH = r"D:\STATS NZ DATASET\employment-indicators-july-2026\employment-indicators-july-2026-csv-tables.csv"
DATASET_LABEL = "Employment Indicators - July 2026"

print("=" * 85)
print(f"DATA QUALITY AUDIT REPORT: {DATASET_LABEL.upper()}")
print("=" * 85)

# Verify file existence before processing
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"❌ File not found at path: {FILE_PATH}")

# -----------------------------------------------------------------------------
# 2. DEFINING STATS NZ NA PATTERNS & EFFICIENT DATA LOADING
# -----------------------------------------------------------------------------
# Standard missing/suppressed value codes used across Stats NZ files
NA_PATTERNS = [
    "n/a",
    "N/A",
    "NA",
    "null",
    "NULL",
    "None",
    "",
    " ",
    "..",
    "C",
    "S",
]

file_size_mb = os.path.getsize(FILE_PATH) / (1024 * 1024)
print(f"📁 Target File Path    : {FILE_PATH}")
print(f"📦 Disk Allocation     : {file_size_mb:.2f} MB")
print("⏳ Loading CSV into memory with optimized parser...")

# Ingest CSV with strict NA detection and disable chunk warnings
df = pd.read_csv(
    FILE_PATH,
    na_values=NA_PATTERNS,
    keep_default_na=True,
    low_memory=False,
)



DATA QUALITY AUDIT REPORT: EMPLOYMENT INDICATORS - JULY 2026
📁 Target File Path    : D:\STATS NZ DATASET\employment-indicators-july-2026\employment-indicators-july-2026-csv-tables.csv
📦 Disk Allocation     : 4.45 MB
⏳ Loading CSV into memory with optimized parser...


In [2]:
# -----------------------------------------------------------------------------
# 3. HIGH-LEVEL DATASET METRICS AUDIT
# -----------------------------------------------------------------------------
total_rows = len(df)
total_cols = df.shape[1]
duplicate_rows = df.duplicated().sum()
duplicate_pct = (duplicate_rows / total_rows) * 100 if total_rows > 0 else 0
memory_usage_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)

print("\n" + "-" * 85)
print("1. OVERVIEW METRICS")
print("-" * 85)
print(f"📊 Total Records (Rows) : {total_rows:,}")
print(f"📋 Total Attributes (Cols): {total_cols}")
print(
    f"🔁 Duplicate Rows       : {duplicate_rows:,} ({duplicate_pct:.2f}% of total)"
)
print(f"💾 In-Memory Usage      : {memory_usage_mb:.2f} MB\n")




-------------------------------------------------------------------------------------
1. OVERVIEW METRICS
-------------------------------------------------------------------------------------
📊 Total Records (Rows) : 34,582
📋 Total Attributes (Cols): 14
🔁 Duplicate Rows       : 0 (0.00% of total)
💾 In-Memory Usage      : 21.44 MB



In [3]:
# -----------------------------------------------------------------------------
# 4. COLUMN-LEVEL DATA HYGIENE CHECK
# -----------------------------------------------------------------------------
column_audit = []

for col in df.columns:
    col_data = df[col]

    # Calculate missing value counts and percentages
    null_count = col_data.isna().sum()
    null_pct = (null_count / total_rows) * 100 if total_rows > 0 else 0

    # Capture data type and distinct value counts
    inferred_dtype = str(col_data.dtype)
    unique_count = col_data.nunique(dropna=True)

    # Detect dirty numeric strings inside object/text columns
    invalid_numeric_count = 0
    if inferred_dtype == "object":
        numeric_conversion = pd.to_numeric(col_data, errors="coerce")
        invalid_numeric_count = (
            col_data.notna() & numeric_conversion.isna()
        ).sum()

    # Extract sample values for quick verification
    sample_values = col_data.dropna().unique()[:3]
    sample_str = (
        ", ".join(map(str, sample_values)) if len(sample_values) > 0 else "N/A"
    )

    column_audit.append({
        "Column Name": col,
        "Data Type": inferred_dtype,
        "Null Count": null_count,
        "Null %": round(null_pct, 2),
        "Unique Values": unique_count,
        "Dirty Numeric Strings": invalid_numeric_count,
        "Sample Values": sample_str,
    })

# Convert audit results to DataFrame for HTML display in Jupyter
audit_df = pd.DataFrame(column_audit)

print("-" * 85)
print("2. COLUMN HYGIENE & QUALITY BREAKDOWN")
print("-" * 85)
display(audit_df)



-------------------------------------------------------------------------------------
2. COLUMN HYGIENE & QUALITY BREAKDOWN
-------------------------------------------------------------------------------------


,Column Name,Data Type,Null Count,Null %,Unique Values,Dirty Numeric Strings,Sample Values
0,Series_reference,object,0,0.00,326,34582,"MEIM.S1WA, MEIM.S1WS, MEIM.S1WT"
1,Period,float64,0,0.00,328,0,"1999.04, 1999.05, 1999.06"
2,Data_value,float64,50,0.14,27424,0,"80267.0, 70803.0, 65792.0"
3,Suppressed,float64,34582,100.00,0,0,N/A
4,STATUS,object,50,0.14,3,34532,"F, P, R"
5,UNITS,object,0,0.00,2,34582,"Number, Value"
6,Magnitude,int64,0,0.00,2,0,"0, 6"
7,Subject,object,0,0.00,1,34582,Employment indicators - MEI
8,Group,object,0,0.00,6,34582,"High level industry by variable, Industry by v..."
9,Series_title_1,object,0,0.00,3,34582,"Filled jobs, Earnings - cash, Earnings - accrued"


In [4]:
# -----------------------------------------------------------------------------
# 5. AUTOMATED ANOMALY & DATA HEALTH WARNINGS
# -----------------------------------------------------------------------------
print("\n" + "-" * 85)
print("3. AUTOMATED ANOMALY HIGHLIGHTS")
print("-" * 85)

# High null threshold warning (>40%)
high_null_cols = audit_df[audit_df["Null %"] > 40.0]["Column Name"].tolist()
if high_null_cols:
    print(
        f"⚠️  High Missing Data Alert (>40% nulls detected): {high_null_cols}"
    )
else:
    print("✅ High Completeness: No columns exceed 40% missing data.")

# Duplicate check
if duplicate_rows > 0:
    print(
        f"⚠️  Duplicate Alert: Found {duplicate_rows:,} exact duplicate rows."
    )
else:
    print("✅ Zero Duplicate Records Found.")

# Mixed type numeric check
cols_with_dirty_nums = audit_df[audit_df["Dirty Numeric Strings"] > 0][
    "Column Name"
].tolist()
if cols_with_dirty_nums:
    print(
        f"⚠️  String-Numeric Mixed Types Warning in Columns: {cols_with_dirty_nums}"
    )
else:
    print("✅ Numerical Integrity Passed: All text columns are clean strings.")


-------------------------------------------------------------------------------------
3. AUTOMATED ANOMALY HIGHLIGHTS
-------------------------------------------------------------------------------------
⚠️  High Missing Data Alert (>40% nulls detected): ['Suppressed', 'Series_title_5']
✅ Zero Duplicate Records Found.
⚠️  String-Numeric Mixed Types Warning in Columns: ['Series_reference', 'STATUS', 'UNITS', 'Subject', 'Group', 'Series_title_1', 'Series_title_2', 'Series_title_3', 'Series_title_4']
